# Notebook 1 — Diagnóstico de Overfitting com MLP

**Objetivo didático:** identificar sinais de overfitting em um problema de classificação supervisionada e decidir quais intervenções fazem sentido.

## O que você deve aprender
- Diferenciar desempenho de treino e validação.
- Reconhecer evidências de memorização.
- Alterar o código de forma justificada, não apenas por tentativa e erro.
- Comparar o efeito de regularização L2 e dropout.

## Tarefa
1. Execute o notebook sem alterações.
2. Observe curvas e métricas.
3. Diagnostique o problema.
4. Modifique o modelo e/ou o otimizador.
5. Registre o que mudou e por quê.

## Perguntas orientadoras
- O modelo generaliza bem?
- Que evidências sustentam sua conclusão?
- Qual seria sua primeira intervenção?
- Em que ponto a regularização ajuda de fato?

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Verificar se a GPU está disponível
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f'Usando o dispositivo: {device}')

## 1. Geração do dataset

Neste cenário, o conjunto de dados é propositalmente **pequeno** para aumentar a chance de overfitting.

In [ ]:
X, y = make_classification(
    n_samples=240,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_classes=2,
    class_sep=1.0,
    flip_y=0.03,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.35, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)

X_train_t.shape, X_val_t.shape

## 2. Modelo inicial

**Importante:** o modelo abaixo foi montado para ter **capacidade alta** em relação ao tamanho do dataset.

### Sua análise
Antes de alterar qualquer coisa, pergunte:
- A arquitetura parece grande demais?
- Há algum mecanismo explícito de regularização?

In [ ]:
class MLPOverfit(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

model = MLPOverfit(X_train_t.shape[1]).to(device)
model

In [ ]:
criterion = nn.CrossEntropyLoss()

# CENÁRIO INICIAL: sem regularização explícita
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# TODO:
# Teste versões com weight_decay, por exemplo:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == yb).sum().item()
            total += xb.size(0)

    return total_loss / total, total_correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=120):
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, criterion)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} | "
                f"train_acc={tr_acc:.4f} val_acc={va_acc:.4f}"
            )

    return history

In [ ]:
history = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=120)

In [ ]:
def plot_history(history, title="Histórico de treino"):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train loss")
    plt.plot(epochs, history["val_loss"], label="Val loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_acc"], label="Train acc")
    plt.plot(epochs, history["val_acc"], label="Val acc")
    plt.xlabel("Época")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.show()

plot_history(history, title="Modelo inicial")

## 3. Diagnóstico

Preencha antes de alterar o código:

- **Sinal principal observado:**  
- **Há gap entre treino e validação?**  
- **O modelo parece subajustado, equilibrado ou sobreajustado?**  
- **Qual alteração você faria primeiro? Por quê?**

## 4. Intervenções sugeridas

Teste uma alteração por vez:
1. Adicionar `weight_decay`.
2. Inserir `Dropout`.
3. Reduzir a largura das camadas.
4. Comparar combinações.

**Não mude tudo de uma vez.** Você deve conseguir justificar o efeito de cada intervenção.

In [ ]:
# ÁREA DE EXPERIMENTAÇÃO - Aqui você pode testar novas hipóteses para melhorar a generalização do modelo.
# Copie e adapte este bloco para testar novas hipóteses.

class MLPRegularized(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

model_exp = MLPRegularized(X_train_t.shape[1], dropout=0.3).to(device)
optimizer_exp = torch.optim.Adam(model_exp.parameters(), lr=1e-3, weight_decay=1e-4)

history_exp = train_model(model_exp, train_loader, val_loader, criterion, optimizer_exp, epochs=120)
plot_history(history_exp, title="Experimento do aluno")

## 5. Registro final

Monte uma tabela com suas execuções:

| Configuração | Train Acc Final | Val Acc Final | Gap | Interpretação |
|---|---:|---:|---:|---|
| Baseline |  |  |  |  |
| + L2 |  |  |  |  |
| + Dropout |  |  |  |  |
| L2 + Dropout |  |  |  |  |

## Conclusão
Responda:
- Qual configuração teve melhor capacidade de generalização?
- Qual configuração apenas reduziu desempenho sem melhorar generalização?
